In [ ]:
#install duckDB
!pip install duckdb
!pip install pandas
!pip install psycopg2-binary

In [ ]:
!apt-get update -y

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,164 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,002 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 11.3 MB in 8s (1,337 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (s

In [ ]:
!apt-get -y install postgresql postgresql-contrib


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
postgresql is already the newest version (14+238).
postgresql-contrib is already the newest version (14+238).
0 upgraded, 0 newly installed, 0 to remove and 139 not upgraded.


In [ ]:
#starting postgre sql
!service postgresql start
!sudo -u postgres psql -c "CREATE DATABASE tpch;"

 * Starting PostgreSQL 14 database server
   ...done.
ERROR:  database "tpch" already exists


In [ ]:
import duckdb

duck_con = duckdb.connect()


In [ ]:
#loding tpc-h
duck_con.execute("INSTALL tpch")
duck_con.execute("LOAD tpch")

In [ ]:
#generating 5 gb data base using DBgen
duck_con.execute("CALL dbgen(sf=5)")

In [ ]:
duck_con.execute("SHOW TABLES").fetchall()

In [ ]:
duck_con.execute("SELECT * FROM region LIMIT 5").fetchdf()

In [ ]:
duck_con.execute("SELECT * FROM nation LIMIT 5").fetchdf()

In [ ]:
duck_con.execute("SELECT * FROM supplier LIMIT 5").fetchdf()

In [ ]:
duck_con.execute("SELECT * FROM customer LIMIT 5").fetchdf()

In [ ]:
duck_con.execute("SELECT * FROM orders LIMIT 5").fetchdf()

In [ ]:
duck_con.execute("SELECT * FROM lineitem LIMIT 10").fetchdf()

In [ ]:
duck_con.execute("SELECT COUNT(*) FROM orders").fetchall()

In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'tpchDB';" #setting paswoord

In [ ]:
import psycopg2

pg_con = psycopg2.connect(
    database="postgres",   # first connect to default DB
    user="postgres",
    password="tpchDB",
    host="localhost",
    port="5432"
)

pg_cur = pg_con.cursor()

print("Connected successfully")

In [ ]:
#lineitem
pg_cur.execute("""
CREATE TABLE lineitem(
l_orderkey INTEGER,
l_partkey INTEGER,
l_suppkey INTEGER,
l_linenumber INTEGER,
l_quantity DOUBLE PRECISION,
l_extendedprice DOUBLE PRECISION,
l_discount DOUBLE PRECISION,
l_tax DOUBLE PRECISION,
l_returnflag CHAR(1),
l_linestatus CHAR(1),
l_shipdate DATE,
l_commitdate DATE,
l_receiptdate DATE,
l_shipinstruct TEXT,
l_shipmode TEXT,
l_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE region(
r_regionkey INTEGER,
r_name TEXT,
r_comment TEXT
);
""")
pg_con.commit()

In [ ]:
#pg_con.rollback()

In [ ]:
pg_cur.execute("""
CREATE TABLE nation(
n_nationkey INTEGER,
n_name TEXT,
n_regionkey INTEGER,
n_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE supplier(
s_suppkey INTEGER,
s_name TEXT,
s_address TEXT,
s_nationkey INTEGER,
s_phone TEXT,
s_acctbal DOUBLE PRECISION,
s_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE customer(
c_custkey INTEGER,
c_name TEXT,
c_address TEXT,
c_nationkey INTEGER,
c_phone TEXT,
c_acctbal DOUBLE PRECISION,
c_mktsegment TEXT,
c_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE part(
p_partkey INTEGER,
p_name TEXT,
p_mfgr TEXT,
p_brand TEXT,
p_type TEXT,
p_size INTEGER,
p_container TEXT,
p_retailprice DOUBLE PRECISION,
p_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE partsupp(
ps_partkey INTEGER,
ps_suppkey INTEGER,
ps_availqty INTEGER,
ps_supplycost DOUBLE PRECISION,
ps_comment TEXT
);
""")
pg_con.commit()

In [ ]:
pg_cur.execute("""
CREATE TABLE orders(
o_orderkey INTEGER,
o_custkey INTEGER,
o_orderstatus CHAR(1),
o_totalprice DOUBLE PRECISION,
o_orderdate DATE,
o_orderpriority TEXT,
o_clerk TEXT,
o_shippriority INTEGER,
o_comment TEXT
);
""")

pg_con.commit()

In [ ]:
#exporting data
duck_con.execute("COPY region TO 'region.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY nation TO 'nation.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY supplier TO 'supplier.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY customer TO 'customer.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY part TO 'part.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY partsupp TO 'partsupp.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY orders TO 'orders.csv' (HEADER, DELIMITER ',')")
duck_con.execute("COPY lineitem TO 'lineitem.csv' (HEADER, DELIMITER ',')")

In [ ]:
tables = [
    "region",
    "nation",
    "supplier",
    "customer",
    "part",
    "partsupp",
    "orders",
    "lineitem"
]

for table in tables:
    with open(f"{table}.csv","r") as f:
        pg_cur.copy_expert(
            f"COPY {table} FROM STDIN WITH CSV HEADER",
            f
        )
    pg_con.commit()

print("All tables loaded into PostgreSQL")

In [ ]:
tables = [
    "region",
    "nation",
    "supplier",
    "customer",
    "part",
    "partsupp",
    "orders",
    "lineitem"
]

for table in tables:
    pg_cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = pg_cur.fetchone()[0]
    print(f"{table}:", count)

In [ ]:
#run only when region > 5 and nation > 25 in previous command
# pg_cur.execute("TRUNCATE TABLE region, nation")
# pg_con.commit()

In [ ]:
# #run only when region > 5 and nation > 25 in previous command
# tables = ["region","nation"]

# for table in tables:
#     with open(f"{table}.csv","r") as f:
#         pg_cur.copy_expert(
#             f"COPY {table} FROM STDIN WITH CSV HEADER",
#             f
#         )
#     pg_con.commit()

# print("Region and Nation reloaded")

In [ ]:
tables = ["region","nation","supplier","customer","part","partsupp","orders","lineitem"]

for table in tables:
    pg_cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(table, pg_cur.fetchone()[0])

In [ ]:
# #top 8 queries
# queries = {
# "Q1": """
# SELECT
# l_returnflag,
# l_linestatus,
# SUM(l_quantity) AS sum_qty,
# SUM(l_extendedprice) AS sum_price
# FROM lineitem
# GROUP BY l_returnflag,l_linestatus
# """,

# "Q3": """
# SELECT
# l_orderkey,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM lineitem
# GROUP BY l_orderkey
# ORDER BY revenue DESC
# LIMIT 10
# """,

# "Q5": """
# SELECT
# n_name,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM customer
# JOIN "orders" ON c_custkey=o_custkey
# JOIN lineitem ON l_orderkey=o_orderkey
# JOIN supplier ON l_suppkey=s_suppkey
# JOIN nation ON s_nationkey=n_nationkey
# GROUP BY n_name
# """,

# "Q6": """
# SELECT
# SUM(l_extendedprice*l_discount) AS revenue
# FROM lineitem
# WHERE l_discount BETWEEN 0.05 AND 0.07
# AND l_quantity < 24
# """,

# "Q10": """
# SELECT
# c_custkey,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM customer
# JOIN "orders" ON c_custkey=o_custkey
# JOIN lineitem ON l_orderkey=o_orderkey
# GROUP BY c_custkey
# ORDER BY revenue DESC
# LIMIT 20
# """,

# "Q12": """
# SELECT
# l_shipmode,
# COUNT(*) AS order_count
# FROM lineitem
# GROUP BY l_shipmode
# """,

# "Q14": """
# SELECT
# SUM(l_extendedprice*(1-l_discount)) AS promo_revenue
# FROM lineitem
# JOIN part ON l_partkey=p_partkey
# """,

# "Q18": """
# SELECT
# c_name,
# o_orderkey,
# SUM(l_quantity)
# FROM customer
# JOIN "orders" ON c_custkey=o_custkey
# JOIN lineitem ON o_orderkey=l_orderkey
# GROUP BY c_name,o_orderkey
# LIMIT 20
# """
# }

In [ ]:
# queries.update({

# "Q2": """
# SELECT
# s_suppkey,
# s_name,
# n_name,
# ps_supplycost
# FROM supplier
# JOIN nation ON s_nationkey=n_nationkey
# JOIN partsupp ON s_suppkey=ps_suppkey
# ORDER BY ps_supplycost
# LIMIT 10
# """,

# "Q4": """
# SELECT
# o_orderpriority,
# COUNT(*) AS order_count
# FROM "orders"
# GROUP BY o_orderpriority
# ORDER BY o_orderpriority
# """,

# "Q7": """
# SELECT
# supp_nation,
# cust_nation,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM
# (
# SELECT
# n1.n_name AS supp_nation,
# n2.n_name AS cust_nation,
# l_extendedprice,
# l_discount
# FROM supplier
# JOIN nation n1 ON s_nationkey=n1.n_nationkey
# JOIN lineitem ON l_suppkey=s_suppkey
# JOIN "orders" ON o_orderkey=l_orderkey
# JOIN customer ON c_custkey=o_custkey
# JOIN nation n2 ON c_nationkey=n2.n_nationkey
# ) AS shipping
# GROUP BY supp_nation,cust_nation
# """,

# "Q8": """
# SELECT
# n_name,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM part
# JOIN lineitem ON p_partkey=l_partkey
# JOIN "orders" ON o_orderkey=l_orderkey
# JOIN customer ON c_custkey=o_custkey
# JOIN nation ON c_nationkey=n_nationkey
# GROUP BY n_name
# """,

# "Q9": """
# SELECT
# n_name,
# SUM(l_extendedprice*(1-l_discount)-ps_supplycost*l_quantity) AS profit
# FROM lineitem
# JOIN supplier ON l_suppkey=s_suppkey
# JOIN nation ON s_nationkey=n_nationkey
# JOIN partsupp ON ps_partkey=l_partkey
# GROUP BY n_name
# """,

# "Q11": """
# SELECT
# ps_partkey,
# SUM(ps_supplycost*ps_availqty) AS value
# FROM partsupp
# GROUP BY ps_partkey
# ORDER BY value DESC
# LIMIT 10
# """,

# "Q13": """
# SELECT
# c_custkey,
# COUNT(o_orderkey) AS order_count
# FROM customer
# LEFT JOIN "orders" ON c_custkey=o_custkey
# GROUP BY c_custkey
# """,

# "Q15": """
# SELECT
# l_suppkey,
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM lineitem
# GROUP BY l_suppkey
# ORDER BY revenue DESC
# LIMIT 10
# """,

# "Q16": """
# SELECT
# p_brand,
# COUNT(DISTINCT ps_suppkey) AS supplier_count
# FROM part
# JOIN partsupp ON p_partkey=ps_partkey
# GROUP BY p_brand
# """,

# "Q17": """
# SELECT
# SUM(l_extendedprice)/7.0 AS avg_yearly
# FROM lineitem
# JOIN part ON p_partkey=l_partkey
# """,

# "Q19": """
# SELECT
# SUM(l_extendedprice*(1-l_discount)) AS revenue
# FROM lineitem
# JOIN part ON p_partkey=l_partkey
# WHERE l_quantity BETWEEN 1 AND 30
# """,

# "Q20": """
# SELECT
# s_name
# FROM supplier
# JOIN nation ON s_nationkey=n_nationkey
# WHERE n_name='CANADA'
# LIMIT 20
# """,

# "Q21": """
# SELECT
# s_name,
# COUNT(*) AS numwait
# FROM supplier
# JOIN lineitem ON s_suppkey=l_suppkey
# GROUP BY s_name
# ORDER BY numwait DESC
# LIMIT 10
# """,

# "Q22": """
# SELECT
# c_nationkey,
# COUNT(*) AS numcust,
# SUM(c_acctbal) AS totacctbal
# FROM customer
# GROUP BY c_nationkey
# """
# })

In [ ]:
#pg_con.rollback()

In [ ]:
# !mkdir -p queries

In [ ]:
#importing official queries
!wget https://github.com/gregrahn/tpch-kit/archive/refs/heads/master.zip

In [ ]:
!unzip master.zip

In [ ]:
#copying quesries
!mkdir -p queries
!cp tpch-kit-master/dbgen/queries/*.sql queries/

In [ ]:
!ls queries

In [ ]:
#pg_con.rollback()

In [ ]:
# #loding queries in dictionary
queries = {}

for i in range(22):
    q = duck_con.execute(
        f"SELECT query FROM tpch_queries() LIMIT 1 OFFSET {i}"
    ).fetchone()[0]

    queries[f"Q{i+1}"] = q

print("Loaded", len(queries), "official TPC-H queries")

In [ ]:
for k in queries:
    queries[k] = queries[k].replace("orders", '"orders"')

In [ ]:
#INDEXING
pg_cur.execute("""
CREATE INDEX IF NOT EXISTS idx_lineitem_orderkey ON lineitem(l_orderkey);
CREATE INDEX IF NOT EXISTS idx_orders_orderkey ON "orders"(o_orderkey);
CREATE INDEX IF NOT EXISTS idx_orders_custkey ON "orders"(o_custkey);
CREATE INDEX IF NOT EXISTS idx_customer_custkey ON customer(c_custkey);
CREATE INDEX IF NOT EXISTS idx_supplier_suppkey ON supplier(s_suppkey);
CREATE INDEX IF NOT EXISTS idx_nation_nationkey ON nation(n_nationkey);
""")

pg_con.commit()

In [ ]:
pg_cur.execute("ANALYZE")
pg_con.commit()

In [ ]:
#Adding timeout
pg_cur.execute("SET statement_timeout = 300000")  # 5 min

In [ ]:
#testing
duck_con.execute(queries["Q1"]).fetchall()

pg_cur.execute(queries["Q1"])
pg_cur.fetchall()

In [ ]:
#benchmark
import time

duck_times = {}
pg_times = {}

for name, query in queries.items():

    print(f"Running {name}...")

    # DuckDB
    start = time.time()
    duck_con.execute(query).fetchall()
    duck_times[name] = time.time() - start

    # PostgreSQL (with timeout)
    try:
        start = time.time()
        pg_cur.execute(query)
        pg_cur.fetchall()
        pg_times[name] = time.time() - start
    except:
        pg_times[name] = "TIMEOUT"
        pg_con.rollback()

    print(name, "DuckDB:", duck_times[name], "PostgreSQL:", pg_times[name])

Running Q1...
Q1 DuckDB: 1.780139446258545 PostgreSQL: 36.32816457748413
Running Q2...
Q2 DuckDB: 0.42532896995544434 PostgreSQL: TIMEOUT
Running Q3...
Q3 DuckDB: 1.2616322040557861 PostgreSQL: 47.05639600753784
Running Q4...
Q4 DuckDB: 1.1585791110992432 PostgreSQL: 54.32243061065674
Running Q5...
Q5 DuckDB: 0.8924763202667236 PostgreSQL: 59.08518314361572
Running Q6...
Q6 DuckDB: 0.3254578113555908 PostgreSQL: 28.256109952926636
Running Q7...
Q7 DuckDB: 0.8510298728942871 PostgreSQL: 49.821746826171875
Running Q8...
Q8 DuckDB: 0.7980461120605469 PostgreSQL: 44.07812690734863
Running Q9...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q9 DuckDB: 4.754103660583496 PostgreSQL: 1660.896042585373
Running Q10...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q10 DuckDB: 3.632733106613159 PostgreSQL: 55.05178499221802
Running Q11...
Q11 DuckDB: 0.4523274898529053 PostgreSQL: 15.105565786361694
Running Q12...
Q12 DuckDB: 1.1117267608642578 PostgreSQL: 46.943806171417236
Running Q13...


ParserException: Parser Error: syntax error at or near ""orders""

In [ ]:
#pg_con.rollback()